# 04 — Rebuild and package reports

This notebook **does not install MPI, compile C++, or rerun BFS**. It selects the best available completed matrix (`full_matrix` first, otherwise `pilot`), verifies its saved manifests/CSVs/config checkpoints, then regenerates the charts, Markdown report, two-page DOCX and PowerPoint.

It packages the report export and supporting CSVs in `04_<run>_reports_results.zip` only if every report-generation command and final output check succeeds. On failure, no ZIP is generated.

In [ ]:
from pathlib import Path
import os, sys, shutil, json, traceback
from datetime import datetime, timezone
from google.colab import drive

drive.mount('/content/drive')
WORKSPACE = Path('/content/drive/MyDrive/Distributed_BFS_MPI_Colab')
SOURCE_DIR = WORKSPACE / 'Distributed_BFS_MPI'
WORK_DIR = Path('/content/Distributed_BFS_MPI')
HELPER_PATH = WORKSPACE / 'colab_helpers.py'

# Clear previous output archives before importing any project helper or validating
# prerequisites. A failed rerun must never leave an older ZIP looking current.
for _name in ('04_full_matrix_reports_results.zip', '04_pilot_reports_results.zip'):
    if (WORKSPACE / 'downloads').exists():
        (WORKSPACE / 'downloads' / _name).unlink(missing_ok=True)
    (Path('/content') / _name).unlink(missing_ok=True)

sys.path.insert(0, str(WORKSPACE))
import colab_helpers as cb
for _name in ('04_full_matrix_reports_results.zip', '04_pilot_reports_results.zip'):
    cb.clear_previous_archive(WORKSPACE, _name)
if not (SOURCE_DIR / 'src' / 'distributed_bfs.cpp').is_file():
    raise FileNotFoundError(f'Missing source project: {SOURCE_DIR}')

# Automatically select the newest intended completed stage: full matrix first, pilot fallback.
selected = None
selection_errors = {}
for name in ('full_matrix', 'pilot'):
    candidate = WORKSPACE / 'runs' / name
    try:
        info = cb.require_matrix_complete(candidate, SOURCE_DIR)
        selected = (name, candidate, info)
        break
    except Exception as exc:
        selection_errors[name] = f'{type(exc).__name__}: {exc}'
if selected is None:
    raise RuntimeError('No completed benchmark matrix is available. Run Notebook 02, then Notebook 03. Selection diagnostics: ' + json.dumps(selection_errors, indent=2))

RUN_NAME, INPUT_RUN_DIR, MATRIX_INFO = selected
CSV_DIR = INPUT_RUN_DIR / 'csv'
EXPORT_DIR = WORKSPACE / 'exports' / f'reports_{RUN_NAME}'
ARCHIVE_NAME = f'04_{RUN_NAME}_reports_results.zip'
cb.clear_previous_archive(WORKSPACE, ARCHIVE_NAME)
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
for child in ('csv', 'plots', 'report', 'presentation', 'metadata'):
    (EXPORT_DIR / child).mkdir(parents=True, exist_ok=True)
for name in ('summary.csv', 'per_level.csv', 'rank_metrics.csv'):
    shutil.copy2(CSV_DIR / name, EXPORT_DIR / 'csv' / name)
if (INPUT_RUN_DIR / 'environment.txt').is_file():
    shutil.copy2(INPUT_RUN_DIR / 'environment.txt', EXPORT_DIR / 'metadata' / 'environment.txt')
shutil.copy2(INPUT_RUN_DIR / 'RUN_COMPLETE.json', EXPORT_DIR / 'metadata' / 'input_RUN_COMPLETE.json')
shutil.copy2(INPUT_RUN_DIR / 'run_config.json', EXPORT_DIR / 'metadata' / 'input_run_config.json')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(SOURCE_DIR, WORK_DIR)
print('Selected benchmark stage:', RUN_NAME)
print('Input results:', INPUT_RUN_DIR)
print('Report export:', EXPORT_DIR)
print('Validated BFS rows:', MATRIX_INFO['bfs_summary_rows'])

## Generate the report artifacts and create ZIP only after every output passes

The raw benchmark files in `runs/<stage>/` are not modified. New charts and documents are written to a separate `exports/reports_<stage>/` folder.

In [ ]:
try:
    # Report dependencies only; this notebook deliberately does not touch MPI.
    cb.run_stream([
        sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK_DIR / 'requirements.txt')
    ], timeout=900, check=True)

    cb.run_stream([
        sys.executable, str(WORK_DIR / 'scripts' / 'analyze_results.py'),
        '--input-dir', str(EXPORT_DIR / 'csv'),
        '--output-dir', str(EXPORT_DIR / 'plots'),
    ], cwd=WORK_DIR, timeout=900, check=True)

    environment_file = INPUT_RUN_DIR / 'environment.txt'
    environment_arg = str(environment_file) if environment_file.is_file() else ''
    cb.run_stream([
        sys.executable, str(WORK_DIR / 'scripts' / 'generate_report.py'),
        '--input-dir', str(EXPORT_DIR / 'csv'),
        '--output', str(EXPORT_DIR / 'report' / 'Measured_Analysis_Report.md'),
        '--environment', environment_arg,
    ], cwd=WORK_DIR, timeout=900, check=True)

    cb.run_stream([
        sys.executable, str(WORK_DIR / 'scripts' / 'generate_report_docx.py'),
        '--input-dir', str(EXPORT_DIR / 'csv'),
        '--plots-dir', str(EXPORT_DIR / 'plots'),
        '--output', str(EXPORT_DIR / 'report' / 'Two_Page_Report.docx'),
        '--environment', environment_arg,
    ], cwd=WORK_DIR, timeout=900, check=True)

    cb.run_stream([
        sys.executable, str(WORK_DIR / 'scripts' / 'make_presentation.py'),
        '--results-dir', str(EXPORT_DIR),
        '--output', str(EXPORT_DIR / 'presentation' / 'Distributed_BFS_Results_Presentation.pptx'),
    ], cwd=WORK_DIR, timeout=900, check=True)

    # Independent final artifact checks; a placeholder or missing chart is a hard failure.
    expected_datasets = MATRIX_INFO['datasets']
    for dataset in expected_datasets:
        for pattern in ('teps_*.png', 'per_level_*.png', 'collective_request_*.png', 'cut_imbalance_*.png', 'remote_pairs_by_rank_*.png'):
            matches = [p for p in (EXPORT_DIR / 'plots').glob(pattern) if dataset in p.stem]
            if not matches or any(p.stat().st_size == 0 for p in matches):
                raise RuntimeError(f'Required plot missing/empty for {dataset}: {pattern}')
    required = [
        EXPORT_DIR / 'plots' / 'summary_aggregated.csv',
        EXPORT_DIR / 'plots' / 'per_level_derived.csv',
        EXPORT_DIR / 'report' / 'Measured_Analysis_Report.md',
        EXPORT_DIR / 'report' / 'Two_Page_Report.docx',
        EXPORT_DIR / 'presentation' / 'Distributed_BFS_Results_Presentation.pptx',
    ]
    for path in required:
        if not path.is_file() or path.stat().st_size == 0:
            raise RuntimeError(f'Required report output missing or empty: {path}')

    # Verify the DOCX and PPTX can be reopened, not just that they exist.
    from docx import Document
    from pptx import Presentation
    doc = Document(str(EXPORT_DIR / 'report' / 'Two_Page_Report.docx'))
    deck = Presentation(str(EXPORT_DIR / 'presentation' / 'Distributed_BFS_Results_Presentation.pptx'))
    if len(doc.paragraphs) < 5:
        raise RuntimeError('DOCX report appears incomplete (too few paragraphs).')
    if len(deck.slides) < 14:
        raise RuntimeError(f'Presentation appears incomplete: only {len(deck.slides)} slides.')

    completion = {
        'input_run': RUN_NAME,
        'input_bfs_rows': MATRIX_INFO['bfs_summary_rows'],
        'input_configurations': MATRIX_INFO['configurations'],
        'datasets': expected_datasets,
        'charts_verified': sum(1 for p in (EXPORT_DIR / 'plots').glob('*.png') if p.stat().st_size > 0),
        'docx_paragraphs': len(doc.paragraphs),
        'pptx_slides': len(deck.slides),
    }
    cb.write_success_manifest(EXPORT_DIR, '04', {
        'source_fingerprint': cb.source_fingerprint(SOURCE_DIR),
        'completion': completion,
    })
    cb.create_success_zip(EXPORT_DIR, WORKSPACE, ARCHIVE_NAME)
    print('\nREPORT REBUILD COMPLETE.')
    print('Report files validated:', completion)
    print('Automatically downloaded:', ARCHIVE_NAME)

except BaseException as exc:
    (EXPORT_DIR / 'SUCCESS.json').unlink(missing_ok=True)
    (EXPORT_DIR / 'FAILURE.json').write_text(json.dumps({
        'success': False, 'notebook': '04', 'input_run': RUN_NAME,
        'error_type': type(exc).__name__, 'error': str(exc),
        'traceback': traceback.format_exc(),
        'failed_at_utc': datetime.now(timezone.utc).isoformat(),
    }, indent=2), encoding='utf-8')
    print('\nREPORT REBUILD FAILED: no ZIP was created.')
    print('Diagnostics remain in:', EXPORT_DIR)
    raise

## Report export

On success, the notebook automatically downloads `04_<run>_reports_results.zip` and saves a persistent copy under `MyDrive/Distributed_BFS_MPI_Colab/downloads/`. The ZIP contains the report export, plots, slide deck, input CSVs and completion/configuration metadata. The original benchmark folder is left untouched.